# SOLUTIONS

# Gini coding warm-up: 45 minutes
**Shaami's practice notebook.** It follows the job spec: load a client extract, clean it, join it, sense-check it, build the tables that go in a pack, fit a PD model, then check it (Gini, calibration, PSI) and reconcile it against a "SAS" output.

**How to use it**
- Run the setup cell once. It creates fake, deliberately messy data.
- Each exercise cell has a `# TODO` and some `...` to replace. Comments starting `# HINT` tell you which function to reach for.
- Run the **CHECK** cell straight after each exercise. If it prints ✅ you're done; if it errors, read the message.
- Time budget is in each heading. If you're stuck for more than 3 minutes, open the solutions notebook (the job spec says *two hours then ask*; you've got 45 minutes, so make it 3).
- 💬 **Say it out loud** boxes are what you'd say to an interviewer while coding. Practise saying them.

| Section | Minutes |
|---|---|
| 1. Inspect | 5 |
| 2. Clean | 8 |
| 3. Join and reconcile | 7 |
| 4. Summary table for a pack | 6 |
| 5. WoE and IV | 6 |
| 6. Logistic regression and Gini | 7 |
| 7. Calibration and PSI | 6 |
| 8. Bonus: reconcile Python vs SAS | if time |

## 0. Setup: run this, don't edit it

In [30]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 5000

# --- a mortgage book "as of development" ---------------------------------
ltv = np.clip(rng.normal(65, 15, n), 10, 110).round(1)
arrears = rng.choice([0, 1, 2, 3], size=n, p=[0.85, 0.09, 0.04, 0.02])
logit = -4 + 0.035 * (ltv - 60) + 0.9 * arrears          # the "true" risk
default_flag = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

loans = pd.DataFrame({
    "acct_id": np.arange(1, n + 1).astype(float),
    "product": rng.choice(["Fixed", "fixed", "FIXED ", " Tracker", "tracker", "SVR", "svr "], n),
    "ltv": ltv,
    "arrears_months": arrears,
    "open_date": pd.to_datetime("2015-01-01") + pd.to_timedelta(rng.integers(0, 3000, n), unit="D"),
    "default_flag": default_flag,
})
loans["open_date"] = loans["open_date"].dt.strftime("%d/%m/%Y")        # dates arrive as UK text
loans.loc[rng.choice(n, 150, replace=False), "ltv"] = np.nan             # some missing LTVs
loans.loc[[17, 2301], "acct_id"] = np.nan                                # two broken IDs
loans = pd.concat([loans, loans.sample(25, random_state=1)], ignore_index=True)  # duplicate rows

# --- bureau file from another system: IDs stored as TEXT, a few accounts missing ---
keep = rng.random(n) > 0.04
bureau = pd.DataFrame({
    "acct_id": np.arange(1, n + 1)[keep].astype(str),
    "bureau_score": rng.integers(300, 800, keep.sum()),
})

# --- a more recent snapshot for monitoring: LTVs have drifted up (house prices fell) ---
m = 3000
recent = pd.DataFrame({
    "ltv": np.clip(rng.normal(71, 16, m), 10, 120).round(1),
    "arrears_months": rng.choice([0, 1, 2, 3], size=m, p=[0.82, 0.10, 0.05, 0.03]),
})
print("Setup done:", loans.shape, bureau.shape, recent.shape)

Setup done: (5025, 6) (4788, 2) (3000, 2)


## 1. Inspect (5 min)
First thing with any client extract: **look before you touch.** Shape, types, missing values, odd categories.

In [3]:
print(loans.shape)
print(loans.dtypes)
loans.head()

(5025, 6)
acct_id           float64
product            object
ltv               float64
arrears_months      int64
open_date          object
default_flag        int64
dtype: object


,acct_id,product,ltv,arrears_months,open_date,default_flag
0,1.0,Fixed,69.6,0,25/01/2021,0
1,2.0,Tracker,49.4,0,01/08/2016,0
2,3.0,fixed,76.3,0,24/02/2018,0
3,4.0,fixed,79.1,0,10/09/2021,0
4,5.0,fixed,35.7,0,10/01/2020,0


👀 Look at `acct_id`. Its type is **float64**, not int. Why? (Answer in the next exercise.)

In [4]:
missing = loans.isna().sum()
missing

acct_id             2
product             0
ltv               151
arrears_months      0
open_date           0
default_flag        0
dtype: int64

In [5]:
# CHECK 1.2
assert missing["ltv"] == loans["ltv"].isna().sum() and missing["acct_id"] == 2
print("✅ 2 missing IDs - that one NaN is why the whole ID column became float")

✅ 2 missing IDs - that one NaN is why the whole ID column became float


In [ ]:
loans["product"].value_counts() 

product
Fixed       749
svr         748
tracker     732
fixed       719
 Tracker    701
SVR         696
FIXED       680
Name: count, dtype: int64

💬 **Say it out loud:** *"Before cleaning I profile the data: shape, types, missing counts and category spellings. Two things jump out: account ID is float because of missing IDs, and product has seven spellings of three products."*

## 2. Clean (8 min)
Do one thing per step, and **check the row count after each step** so nothing disappears unexplained.

In [7]:
before = len(loans)
loans = loans.drop_duplicates()
print("rows before:", before, "after:", len(loans), "removed:", before - len(loans))

rows before: 5025 after: 5000 removed: 25


In [ ]:
# CHECK 2.1
assert len(loans) == 5000
print("✅ 25 duplicates gone")

In [9]:
loans = loans.dropna(subset=["acct_id"])
loans["acct_id"] = loans["acct_id"].astype("Int64")
print(loans["acct_id"].dtype, len(loans))

Int64 4998


In [11]:
# CHECK 2.2
assert str(loans["acct_id"].dtype) == "Int64" and len(loans) == 4998
print("✅ IDs are whole numbers again; 2 broken records removed (in real life you'd log them and tell the client)")

✅ IDs are whole numbers again; 2 broken records removed (in real life you'd log them and tell the client)


In [ ]:
loans["product"] = loans["product"].str.strip().str.upper()
loans["product"].value_counts()

In [ ]:
# CHECK 2.3
assert set(loans["product"]) == {"FIXED", "TRACKER", "SVR"}
print("✅ 3 clean categories")

In [ ]:
loans["open_date"] = pd.to_datetime(loans["open_date"], format="%d/%m/%Y")
loans["open_date"].min(), loans["open_date"].max()

In [12]:
# CHECK 2.4
assert pd.api.types.is_datetime64_any_dtype(loans["open_date"])
print("✅ dates parsed")

AssertionError: 

In [17]:
loans

,acct_id,product,ltv,arrears_months,open_date,default_flag
0,1.0,Fixed,69.6,0,25/01/2021,0
1,2.0,Tracker,49.4,0,01/08/2016,0
2,3.0,fixed,76.3,0,24/02/2018,0
3,4.0,fixed,79.1,0,10/09/2021,0
4,5.0,fixed,35.7,0,10/01/2020,0
...,...,...,...,...,...,...
5020,2417.0,Fixed,53.8,0,12/12/2022,0
5021,3671.0,Fixed,72.3,1,23/06/2018,0
5022,4839.0,svr,83.0,0,09/01/2015,0
5023,3091.0,Tracker,83.6,2,17/11/2015,0


In [23]:
loans["ltv_missing"] = loans["ltv"].isna().astype(int)

In [24]:
loans.dtypes

acct_id           float64
product            object
ltv               float64
arrears_months      int64
open_date          object
default_flag        int64
ltv_missing         int64
dtype: object

In [26]:
loans["ltv_missing"] = loans["ltv"].isna().astype(int)
loans["ltv"] = loans["ltv"].fillna(loans["ltv"].median())

In [27]:
loans

,acct_id,product,ltv,arrears_months,open_date,default_flag,ltv_missing
0,1.0,Fixed,69.6,0,25/01/2021,0,0
1,2.0,Tracker,49.4,0,01/08/2016,0,0
2,3.0,fixed,76.3,0,24/02/2018,0,0
3,4.0,fixed,79.1,0,10/09/2021,0,0
4,5.0,fixed,35.7,0,10/01/2020,0,0
...,...,...,...,...,...,...,...
5020,2417.0,Fixed,53.8,0,12/12/2022,0,0
5021,3671.0,Fixed,72.3,1,23/06/2018,0,0
5022,4839.0,svr,83.0,0,09/01/2015,0,0
5023,3091.0,Tracker,83.6,2,17/11/2015,0,0


In [28]:
# CHECK 2.5
assert loans["ltv"].isna().sum() == 0 and loans["ltv_missing"].sum() == loans["ltv_missing"].sum() > 0
print("✅ no missing LTV;", loans["ltv_missing"].sum(), "flagged")

AssertionError: 

💬 **Say it out loud:** *"I dedupe, fix the key type, standardise categories, parse dates with an explicit format, and flag missing values before imputing, and I check row counts at every step."*

## 3. Join and reconcile (7 min)
The bureau file stores IDs as **text**. This is where the "101" vs "101.0" bug lives.

In [56]:
loans= loans.dropna(subset=['acct_id'])

In [62]:
loans['acct_id']=loans['acct_id'].astype("Int64")

/var/folders/nr/j3bb4py129n42mzntn2f26g80000gn/T/ipykernel_24028/1872276717.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loans['acct_id']=loans['acct_id'].astype("Int64")


In [64]:
loans['acct_id'].dtypes

Int64Dtype()

In [65]:
bureau['acct_id'].dtypes

Int64Dtype()

In [57]:
loans['acct_id'].isna().value_counts()

acct_id
False    5023
Name: count, dtype: int64

In [42]:
bureau['acct_id'].isna().value_counts()

acct_id
False    4788
Name: count, dtype: int64

In [67]:
bureau["acct_id"] = pd.to_numeric(bureau["acct_id"]).astype("Int64")
df = loans.merge(bureau, on="acct_id", how="left", indicator=True)
print(len(loans), len(df))

5023 5023


In [69]:
df

,acct_id,product,ltv,arrears_months,open_date,default_flag,bureau_score,_merge
0,1,Fixed,69.6,0,25/01/2021,0,675.0,both
1,2,Tracker,49.4,0,01/08/2016,0,NaN,left_only
2,3,fixed,76.3,0,24/02/2018,0,650.0,both
3,4,fixed,79.1,0,10/09/2021,0,449.0,both
4,5,fixed,35.7,0,10/01/2020,0,310.0,both
...,...,...,...,...,...,...,...,...
5018,2417,Fixed,53.8,0,12/12/2022,0,434.0,both
5019,3671,Fixed,72.3,1,23/06/2018,0,598.0,both
5020,4839,svr,83.0,0,09/01/2015,0,693.0,both
5021,3091,Tracker,83.6,2,17/11/2015,0,508.0,both


In [ ]:
assert len(df) == len(loans), "join changed the row count!"
df["_merge"].value_counts()

In [ ]:
unmatched = df[df["_merge"] == "left_only"]
unmatched["acct_id"].head()

1        2
38      40
88      90
115    117
124    126
Name: acct_id, dtype: Int64

In [71]:
unmatched

,acct_id,product,ltv,arrears_months,open_date,default_flag,bureau_score,_merge
1,2,Tracker,49.4,0,01/08/2016,0,NaN,left_only
38,40,Tracker,74.8,3,24/02/2021,1,NaN,left_only
88,90,fixed,71.7,0,31/10/2016,1,NaN,left_only
115,117,fixed,68.5,0,08/03/2016,0,NaN,left_only
124,126,tracker,70.4,0,03/05/2022,0,NaN,left_only
...,...,...,...,...,...,...,...,...
4909,4912,SVR,72.5,0,22/11/2016,0,NaN,left_only
4918,4921,Fixed,59.9,0,15/04/2022,0,NaN,left_only
4942,4945,tracker,62.6,1,28/01/2015,0,NaN,left_only
4956,4959,svr,74.1,0,05/03/2017,0,NaN,left_only


In [72]:
# CHECK 3
assert len(df) == 4998 and 150 < len(unmatched) < 260
print("✅ every account kept;", len(unmatched), "have no bureau score - flag them, don't drop them silently")
df = df.drop(columns="_merge")

AssertionError: 

💬 **Say it out loud:** *"Left join so no accounts drop, validate one-to-one so duplicates can't multiply rows, then reconcile the counts and report the unmatched accounts to the client rather than losing them."*

## 4. Summary table for a client pack (6 min)
The bread and butter: band a variable, group, aggregate.

In [73]:
df["ltv_band"] = pd.cut(df["ltv"], bins=[0, 60, 80, 90, np.inf],
                       labels=["<60", "60-80", "80-90", "90+"], right=False)
df["ltv_band"].value_counts().sort_index()

ltv_band
<60      1797
60-80    2340
80-90     506
90+       229
Name: count, dtype: int64

In [74]:
df

,acct_id,product,ltv,arrears_months,open_date,default_flag,bureau_score,_merge,ltv_band
0,1,Fixed,69.6,0,25/01/2021,0,675.0,both,60-80
1,2,Tracker,49.4,0,01/08/2016,0,NaN,left_only,<60
2,3,fixed,76.3,0,24/02/2018,0,650.0,both,60-80
3,4,fixed,79.1,0,10/09/2021,0,449.0,both,60-80
4,5,fixed,35.7,0,10/01/2020,0,310.0,both,<60
...,...,...,...,...,...,...,...,...,...
5018,2417,Fixed,53.8,0,12/12/2022,0,434.0,both,<60
5019,3671,Fixed,72.3,1,23/06/2018,0,598.0,both,60-80
5020,4839,svr,83.0,0,09/01/2015,0,693.0,both,80-90
5021,3091,Tracker,83.6,2,17/11/2015,0,508.0,both,80-90


In [ ]:
tbl = df.groupby("ltv_band", observed=True).agg(accounts=("default_flag", "size"),
                                                defaults=("default_flag", "sum"))
tbl["default_rate"] = tbl["defaults"] / tbl["accounts"]
tbl

In [ ]:
# CHECK 4
assert tbl["accounts"].sum() == len(df) and tbl["default_rate"].is_monotonic_increasing
print("✅ totals tie out to the book, and default rate rises with LTV (a sense check: does it make business sense?)")

💬 **Say it out loud:** *"Two checks on any summary table: it ties out to the total, and the trend makes business sense. Here, higher LTV means higher default rate."*

## 5. WoE and IV (6 min)
Same bands. WoE = ln(%good / %bad) per band. IV = Σ (%good − %bad) × WoE.

Rule of thumb for IV: <0.02 useless, 0.02–0.1 weak, 0.1–0.3 medium, >0.3 strong (>0.5 check for leakage).

In [ ]:
woe = tbl[["accounts", "defaults"]].copy()
woe["goods"] = woe["accounts"] - woe["defaults"]
woe["pct_good"] = woe["goods"] / woe["goods"].sum()
woe["pct_bad"] = woe["defaults"] / woe["defaults"].sum()
woe["woe"] = np.log(woe["pct_good"] / woe["pct_bad"])
woe["iv_part"] = (woe["pct_good"] - woe["pct_bad"]) * woe["woe"]
iv = woe["iv_part"].sum()
print("IV =", round(iv, 3))
woe

In [ ]:
# CHECK 5
assert woe["woe"].is_monotonic_decreasing and iv > 0.1
print("✅ WoE falls as LTV rises (riskier bands have more bads), IV =", round(iv, 3))

💬 **Say it out loud:** *"WoE is positive where goods are over-represented, negative where bads are. It should move smoothly across bands; if it zig-zags, I'd merge bands. IV tells me how much the variable separates goods from bads on its own."*

## 6. Logistic regression and Gini (7 min)
We'll use **statsmodels** because it gives a SAS-style summary (coefficients, standard errors, p-values). sklearn is fine too, but it hides those.

In [ ]:
import statsmodels.api as sm
X = sm.add_constant(df[["ltv", "arrears_months"]])
model = sm.Logit(df["default_flag"], X).fit(disp=0)
print(model.summary())

👀 **Read the output like a reviewer:**
- **coef**: both should be **positive** (higher LTV, more arrears → more risk). That's the *sign check*.
- **P>|z|**: tiny = the effect is real, not luck.
- The true values I used to make the data were 0.035 and 0.9. How close did you get?

In [ ]:
from sklearn.metrics import roc_auc_score
df["pd"] = model.predict(X)
gini = 2 * roc_auc_score(df["default_flag"], df["pd"]) - 1
print("Gini =", round(gini, 3))

In [ ]:
# CHECK 6
assert (model.params[["ltv", "arrears_months"]] > 0).all() and 0.3 < gini < 0.9
print("✅ signs make sense, Gini =", round(gini, 3))

💬 **Say it out loud:** *"Gini measures ranking: does the model put the defaulters at the risky end. It doesn't tell me whether the PD level is right; that's calibration, next."*

## 7. Calibration and PSI (6 min)

In [ ]:
df["decile"] = pd.qcut(df["pd"], 10, labels=False, duplicates="drop")
calib = df.groupby("decile")[["pd", "default_flag"]].mean()
calib

👀 The two columns should track each other closely. A model fitted on this data is calibrated on average by construction; the real test is on new data over time.

In [ ]:
def psi(expected, actual, n_bins=10):
    edges = np.quantile(expected, np.linspace(0, 1, n_bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e_share = np.histogram(expected, edges)[0] / len(expected)
    a_share = np.histogram(actual, edges)[0] / len(actual)
    e_share, a_share = e_share + 1e-6, a_share + 1e-6
    return np.sum((a_share - e_share) * np.log(a_share / e_share))

ltv_psi = psi(df["ltv"], recent["ltv"])
print("LTV PSI =", round(ltv_psi, 3))

In [ ]:
# CHECK 7
status = "green" if ltv_psi < 0.10 else "amber" if ltv_psi < 0.25 else "red"
print("✅ LTV PSI =", round(ltv_psi, 3), "->", status.upper())
print("   Sanity test: PSI of a dataset against itself should be ~0 ->", round(psi(df["ltv"], df["ltv"]), 6))

💬 **Say it out loud:** *"PSI compares today's distribution with development, using bins fixed on the development data. Under 0.10 green, 0.10 to 0.25 amber, above 0.25 red. LTV has drifted up here, which fits house prices falling. That's a leading signal; Gini and calibration confirm later once defaults come through."*

## 8. Bonus: reconcile Python vs SAS (if time)
Pretend the bank's SAS produced its own PD for every account. Your job: prove they match, and find any that don't.

In [ ]:
# setup for the bonus: run it
sas = df[["acct_id", "pd"]].rename(columns={"pd": "pd_sas"}).copy()
sas["pd_sas"] = sas["pd_sas"] + rng.normal(0, 1e-12, len(sas))   # tiny floating-point noise: fine
sas.loc[sas.index[[10, 400]], "pd_sas"] += 0.002                  # two real breaks
sas = sas.sample(frac=1, random_state=3)                          # SAS output arrives in a different order

In [ ]:
rec = df[["acct_id", "pd"]].merge(sas, on="acct_id", how="outer", validate="one_to_one", indicator=True)
assert (rec["_merge"] == "both").all(), "some accounts are only on one side!"
rec["match"] = np.isclose(rec["pd"], rec["pd_sas"], rtol=0, atol=1e-8)
print(rec["match"].value_counts())
rec.loc[~rec["match"], ["acct_id", "pd", "pd_sas"]]

💬 **Say it out loud:** *"Join on the key, never on row order. Outer join so I'd see accounts on only one side. Compare with an agreed tolerance, because exact equality fails on floating-point noise. Then investigate every break individually."*

---
### Done. Quick self-score
Which sections needed the solutions? Those are your revision list for tomorrow. The most likely live-test asks are sections **2, 3 and 4**, so be quick on those.